# Faruq-v3 — ACMC1 residual error attribution V3 (legacy-checkpoint compatible)

Validation-only. Tidak training dan tidak membuka test. V3 menambahkan compatibility shim untuk checkpoint ACMC1 yang dibuat sebelum atribut ACMC2 `entropy_margin_gates` ada.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, sys, time
from pathlib import Path
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/acmc1-residual-error-audit'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)]
for attempt in range(1,4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)], check=True)
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_acmc1_legacy_checkpoint_compat.py'], cwd=REPO, check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('BRANCH:', BRANCH)
print('COMMIT:', subprocess.check_output(['git','rev-parse','HEAD'], text=True).strip())


In [ ]:
import tarfile, torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root
assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=('bundles/faruq-development-v3-grouped.tar',))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
SEED42 = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt')
SEED123 = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-paired-confirmation-v1/ACMC1/ACMC1_seed123/weights/best.pt')
SEED2026 = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-paired-confirmation-v1/ACMC1/ACMC1_seed2026/weights/best.pt')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-acmc1-residual-error-audit-v1'
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive: archive.extractall('/content', filter='data')
assert (DATA_ROOT / 'data.yaml').is_file(), DATA_ROOT
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
for p in (SEED42, SEED123, SEED2026): assert Path(p).is_file(), p
print('GPU:', torch.cuda.get_device_name(0))
print('OUTPUT:', OUTPUT_ROOT)


In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.analysis.acmc1_residual_error_audit_v3',
    '--seed42-checkpoint', str(SEED42),
    '--seed123-checkpoint', str(SEED123),
    '--seed2026-checkpoint', str(SEED2026),
    '--data-root', str(DATA_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--device', '0',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.run(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(process.stdout, flush=True)
if process.returncode != 0:
    raise RuntimeError(f'Audit V3 gagal dengan exit code {process.returncode}; traceback asli tercetak di atas.')


In [ ]:
import json, pandas as pd
from IPython.display import display
SUMMARY = OUTPUT_ROOT / 'acmc1_residual_error_attribution_v2.json'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['training_executed'] is False
assert result['test_images_accessed'] is False
assert result['test_opened'] is False
global_rows = [{'metric': k, **v} for k,v in result['global_aggregate'].items()]
display(pd.DataFrame(global_rows))
classes = pd.DataFrame(result['per_class_aggregate'])
display(classes[['class_name','map50_95_mean','ap50_mean','ap75_mean','ap95_mean','detection_accessibility_iou50_mean','class_accuracy_given_iou50_match_mean','classification_headroom_iou50_mean','attribution','attribution_seed_agreement']].head(10))
print('ATTRIBUTION COUNTS:', result['attribution_counts'])
print('TOP CONFUSIONS:', result['top_directional_confusions_3seed'][:10])
print('SUMMARY:', SUMMARY)
